# LabOne Q — Lecture 3: Workflows

This notebook introduces the **Workflow DSL**, following the structure of the
[LabOne Q User Manual — Workflows and Tasks](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07_workflow/concepts/index.html):

1. **Tasks** — a plain function wrapped so its inputs/outputs are captured
2. **Workflows** — composing tasks into a repeatable, inspectable pipeline
3. **`compile_experiment` / `run_experiment`** — the built-in tasks every experiment workflow uses
4. **A real example** — Amplitude Rabi as a shipped workflow: the task tree, inspecting a single task's output, `temporary_parameters`, and `show_pulse_sheet` from a workflow result

`05_Workflow_Exercise.ipynb` picks up from here with a live exercise.

Everything below runs in **emulation mode** (`do_emulation=True`), so it can be executed without being
connected to real hardware.

In [ ]:
from laboneq.simple import *
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

from laboneq import workflow
from laboneq.workflow import WorkflowOptions

import numpy as np

## 1. Tasks

A [task](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07_workflow/tutorials/00_tasks.html)
is a plain Python function decorated with `@workflow.task`. On its own it behaves exactly like a normal
function — the decoration only matters once it is called *inside* a workflow (see below), where its
inputs and output get captured for later inspection.

In [ ]:
@workflow.task
def add_one(x: int) -> int:
    return x + 1


@workflow.task
def double(x: int) -> int:
    return x * 2


add_one(4)  # called directly: just a function call, returns 5

## 2. Workflows

A [workflow](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07_workflow/tutorials/01_workflows.html)
is a function decorated with `@workflow.workflow` that calls tasks (and other workflows). Calling it does
**not** run anything yet — it returns a lazy `Workflow` object. Nothing executes until `.run()` is called,
and the result gives access to every task's input and output, not just the final return value.

In [ ]:
@workflow.workflow(name="Toy Workflow")
def toy_workflow(x: int, options: WorkflowOptions | None = None):
    a = add_one(x)
    b = double(a)
    workflow.return_(b)


wf = toy_workflow(5)  # nothing has run yet
result = wf.run()

print("workflow output:", result.output)
print("tasks that ran:", [t.name for t in result.tasks])
print("add_one's own input/output:", result.tasks["add_one"].input, "->", result.tasks["add_one"].output)

## 3. `compile_experiment` and `run_experiment`

Every experiment workflow in `laboneq_applications` (and the one we build below) follows the same shape:
build an `Experiment`, compile it, run it. LabOne Q ships these last two steps as ready-made tasks so you
never have to wrap `session.compile` / `session.run` yourself:

```python
from laboneq.workflow.tasks import compile_experiment, run_experiment
```

- `compile_experiment(session, experiment)` — returns a `CompiledExperiment`
- `run_experiment(session, compiled_experiment)` — returns a `Results` object

Both are ordinary tasks, so a workflow that calls them gets the same inspectable `result.tasks[...]`
entries as any other task.

## 4. A real example — Amplitude Rabi as a workflow

Every standard experiment in `laboneq_applications` is already shipped as a workflow. `amplitude_rabi.experiment_workflow`
is the Amplitude Rabi experiment of Lecture 1, wrapped end to end: build the
experiment, compile it, run it, then (optionally) analyse and update the QPU.

We reuse the two-qubit demo platform from `03_Operations_QPU_Topology.ipynb`. We turn the analysis
step off (`options.do_analysis(False)`) since we only care about the workflow mechanics here, not fitting
emulated data.

In [ ]:
from laboneq_applications.qpu_types.tunable_transmon.demo_qpus import demo_platform
from laboneq_applications.experiments import amplitude_rabi

rabi_platform = demo_platform(n_qubits=2)
rabi_qpu = rabi_platform.qpu
rabi_session = rabi_platform.session(do_emulation=True)

rabi_amplitudes = np.linspace(0.0, 1.0, 11)

rabi_options = amplitude_rabi.experiment_workflow.options()
rabi_options.count(4)
rabi_options.do_analysis(False)

rabi_wf = amplitude_rabi.experiment_workflow(
    session=rabi_session,
    qpu=rabi_qpu,
    qubits="q0",
    amplitudes=rabi_amplitudes,
    options=rabi_options,
)

### The workflow tree

`wf.graph.tree` shows the full structure — every task, and every conditional branch (here, the analysis
branch we just switched off) — before anything has actually run:

In [ ]:
print(rabi_wf.graph.tree)

### Running it, and grabbing a single task's output

`.run()` executes the tree and returns a `WorkflowResult`. Every task's own input/output is still
addressable afterwards, by name — here we pull out just the `CompiledExperiment` from `compile_experiment`,
without needing to re-run or re-compile anything:

In [ ]:
rabi_result = rabi_wf.run()
print([t.name for t in rabi_result.tasks])

rabi_compiled = rabi_result.tasks["compile_experiment"].output
print(type(rabi_compiled))

In [ ]:
plot_simulation(rabi_compiled, length=10000e-9)

### `temporary_parameters`

Most `laboneq_applications` workflows accept a `temporary_parameters` argument: a dict of overrides
applied to a *copy* of the QPU for the duration of that single workflow run, leaving the original `qpu`
untouched. This is the quick way to try "what if this qubit's pi-pulse amplitude were different?" without
mutating anything permanently.

In [ ]:
rabi_wf_override = amplitude_rabi.experiment_workflow(
    session=rabi_session,
    qpu=rabi_qpu,
    qubits="q0",
    amplitudes=rabi_amplitudes,
    temporary_parameters={"q0": {"ge_drive_amplitude_pi": 0.55}},
    options=rabi_options,
)
rabi_result_override = rabi_wf_override.run()

print("original qpu untouched:", rabi_qpu["q0"].parameters.ge_drive_amplitude_pi)

### `show_pulse_sheet` straight from a workflow

The `CompiledExperiment` pulled out of `compile_experiment` is a completely ordinary one — it plugs
straight into `show_pulse_sheet` and `plot_simulation`, exactly as if we had compiled it by hand:

In [ ]:
show_pulse_sheet("rabi_from_workflow", rabi_compiled)
plot_simulation(rabi_compiled, length=rabi_compiled.estimated_runtime, signal_names_to_show=["drive", "measure"])